In [1]:
# Hyperparameter Optimization for Disease Risk Classification
# Install once if needed:
# pip install scikit-optimize

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import randint
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

# Create sample healthcare dataset
X, y = make_classification(
    n_samples=600, n_features=10, n_informative=6,
    n_classes=2, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = RandomForestClassifier(random_state=42)

# 1. Grid Search
grid_parameters = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(model, grid_parameters, cv=3, scoring="accuracy")
grid_search.fit(X_train, y_train)

# 2. Random Search
random_parameters = {
    "n_estimators": randint(50, 250),
    "max_depth": [5, 10, 15, None],
    "min_samples_split": randint(2, 10)
}

random_search = RandomizedSearchCV(
    model, random_parameters, n_iter=10, cv=3,
    scoring="accuracy", random_state=42
)
random_search.fit(X_train, y_train)

# 3. Bayesian Optimization
bayes_parameters = {
    "n_estimators": Integer(50, 250),
    "max_depth": Integer(3, 20),
    "min_samples_split": Integer(2, 10),
    "criterion": Categorical(["gini", "entropy"])
}

bayes_search = BayesSearchCV(
    model, bayes_parameters, n_iter=15, cv=3,
    scoring="accuracy", random_state=42
)
bayes_search.fit(X_train, y_train)

# Evaluate methods
methods = {
    "Grid Search": grid_search,
    "Random Search": random_search,
    "Bayesian Optimization": bayes_search
}

print("Method                    Test Accuracy   Best Hyperparameters")
print("-" * 80)

for name, search in methods.items():
    prediction = search.best_estimator_.predict(X_test)
    accuracy = accuracy_score(y_test, prediction)

    print(f"{name:<25} {accuracy:.4f}          {search.best_params_}")

ModuleNotFoundError: No module named 'skopt'